In [ ]:
# poly_logistic_model
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, PolynomialFeatures, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
import joblib
import os

df = pd.read_excel("Diagnosis3.xlsx")
df['DOB'] = pd.to_datetime(df['DOB'], errors='coerce')
df['DOS From'] = pd.to_datetime(df['DOS From'], errors='coerce')
df['Age'] = ((df['DOS From'] - df['DOB']).dt.days // 365).fillna(-1).astype(int)
df['DOS_Weekday'] = df['DOS From'].dt.weekday.fillna(-1).astype(int)
df['DOS_Month'] = df['DOS From'].dt.month.fillna(-1).astype(int)
if 'Mem.#' in df.columns: df.drop(columns=['Mem.#'], inplace=True)
for c in ['DOB','DOS From']: df.drop(columns=[c], inplace=True)

target_col = 'D1'
df[target_col] = df[target_col].fillna("None")
feature_cols = [c for c in df.columns if c != target_col]
rare_threshold = 20
counts = df[target_col].value_counts()
rare_labels = counts[counts < rare_threshold].index.tolist()
df['D1_grouped'] = df[target_col].apply(lambda x: 'Other' if x in rare_labels else x)
target = 'D1_grouped'

X = df[feature_cols].copy()
cat_cols = X.select_dtypes(include=['object','category']).columns.tolist()
for col in cat_cols:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str).fillna("None"))

X = X.apply(pd.to_numeric, errors='coerce').fillna(-1).astype(np.float32)
y = df[target]
X_trn, X_tst, y_trn, y_tst = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_trn, X_vld, y_trn, y_vld = train_test_split(X_trn, y_trn, test_size=0.2, random_state=42, stratify=y_trn)

poly_lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('clf', LogisticRegression(
        multi_class='multinomial', solver='saga',
        max_iter=1000, penalty='l2', class_weight='balanced'
    ))
])

poly_lr_pipeline.fit(X_trn, y_trn)
y_pred = poly_lr_pipeline.predict(X_tst)
acc = accuracy_score(y_tst, y_pred)
f1 = f1_score(y_tst, y_pred, average='weighted')
print(f"Polynomial Logistic Regression Test Accuracy: {acc:.4f}, Weighted F1: {f1:.4f}")

os.makedirs("models", exist_ok=True)
joblib.dump(poly_lr_pipeline, "models/poly_lr_model.joblib")

C:\Users\elife\anaconda3\envs\myenv\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
